# LCA Lasers4MaaS: pre- and post-joining processes

Unit processes of the chemical and mechanical treatments before and after joining, per material (steel, aluminium).
The functional unit of each process is 1 execution of the process.

The calculation code is in the `l4m_lca` package. This notebook holds the parameters and shows the results.

In [ ]:
import bw2data as bd
from IPython.display import display

from l4m_lca import plots
from l4m_lca.constants import EU27, EUROSTAT_ELEC_PRICE_2025S1, SCENARIOS
from l4m_lca.impacts import BackgroundImpacts
from l4m_lca.lcia import COST_COLUMN, export, scenario_table
from l4m_lca.prepost import chemical_exchanges, mechanical_exchanges, per_material, process_code, process_name
from l4m_lca.project import DEFAULT_INDICATORS, category, ef31_methods, find_activity, method_units, open_project
from l4m_lca.regional import electricity_markets, regional_table

## Parameters

In [ ]:
PROJECT_NAME = "Lasers4MaaS"
ECOINVENT_DATABASE_NAME = "ecoinvent-3.12-cutoff"
RESULTS_DIR = "results"

CHEM_PREPOST_NAME = "Chemical_Pre_and_Post_processes"
MECH_PREPOST_NAME = "Mechanical_Pre_and_Post_processes"

INDICATORS = DEFAULT_INDICATORS

# Background activities: (name, location, reference product or None)
FLOWS = {
    "electricity":        ("market group for electricity, medium voltage", "RER", None),          # kWh
    "water":              ("market group for tap water", "RER", None),                            # kg
    "sodium_hydroxide":   ("market for sodium hydroxide, without water, in 50% solution state", "RER", None),
    "hydrochloric_acid":  ("market for hydrochloric acid, without water, in 30% solution state", "RER", None),
    "nitric_acid":        ("market for nitric acid, without water, in 50% solution state", "RER w/o RU", None),
    # Mechanical flows are PROXIES, all in kg. ecoinvent 3.12 has no "grinding wheel" or
    # "metal working fluid" dataset, and "abrasive blasting, alumina" is a service in m2.
    "abrasive":           ("aluminium oxide production", "RoW", "aluminium oxide, non-metallurgical"),  # alumina grit (corundum)
    "grinding_wheel":     ("market for silicon carbide", "GLO", None),        # abrasive grain of wheels
    "polishing_compound": ("aluminium oxide production", "RoW", "aluminium oxide, non-metallurgical"),  # alumina powder
    "coolant":            ("market for lubricating oil", "RER", None),        # no emulsion dataset in ecoinvent
    "tool_material":      ("market for tungsten carbide powder", "GLO", None),  # cutting tool material
}

# Prices per ecoinvent unit of each flow. INDICATIVE PLACEHOLDER PRICES except electricity:
# replace with supplier quotes. Chemicals are per kg of ACTIVE substance, as modelled
# (ecoinvent "without water" datasets), so they are higher than the diluted commercial price.
COST_FACTORS = {
    "electricity":        0.29,   # EUR/kWh, industrial electricity
    "water":              0.004,  # EUR/kg, tap water with discharge (about 4 EUR/m3)
    "sodium_hydroxide":   0.50,   # EUR/kg
    "hydrochloric_acid":  0.25,   # EUR/kg
    "nitric_acid":        0.45,   # EUR/kg
    "abrasive":           1.50,   # EUR/kg, alumina grit
    "polishing_compound": 1.50,   # EUR/kg, alumina proxy
    "grinding_wheel":     2.50,   # EUR/kg, silicon carbide proxy
    "coolant":            2.00,   # EUR/kg, lubricating oil proxy
    "tool_material":      60.0,   # EUR/kg, tungsten carbide
}

# Regional analysis: industrial electricity price of each country (EUR/kWh)
ELEC_PRICE = EUROSTAT_ELEC_PRICE_2025S1
REGIONAL_BASELINE = "RER"

In [ ]:
# Chemical bath processes
INDUSTRIAL_TANK_SIZES_L = {
    "pickling": 10000,             # 10 m3, standard for joining preparation (ASTM A380)
    "alkaline_degreasing": 7500,   # 7.5 m3, common for degreasing
    "passivation": 7500,           # 7.5 m3, as degreasing
    "chemical_etching": 7500,      # 7.5 m3, as pickling
    "conversion_coating": 3000,    # 3 m3, smaller for dip coating
}
DEFAULT_TANK_SIZE_L = 7500

# Active chemical of each bath
ACTIVE_CHEMICAL = {
    "pickling": "hydrochloric_acid",
    "alkaline_degreasing": "sodium_hydroxide",
    "passivation": "nitric_acid",
    "chemical_etching": "sodium_hydroxide",
}

BATH_DENSITY_KG_PER_L = 1.0      # dilute process baths (water-like)
REPLENISHMENT_PER_H = 0.001      # fraction of the active chemical replenished per hour (losses, degradation)
ROOM_TEMP_C = 20
HEATER_EFFICIENCY = 0.95

# U-values (W/m2.K) of tank walls
U_VALUES = {
    "bare_steel": 9.8,          # uninsulated steel tank
    "stainless_steel": 8.5,
    "fiberglass_5cm": 0.57,     # 5 cm fiberglass insulation
    "polyurethane_5cm": 0.45,
    "typical_insulated": 0.6,
}
TANK_U_VALUE = U_VALUES["typical_insulated"]

In [ ]:
# Pre-/post-joining process parameters with min/max ranges.
# Material-specific fields are keyed by "steel" / "aluminium".
# acid_concentration_% is the mass fraction of active chemical: 0.05 = 5 wt%.
CHEM_PREPOST_PARAMS = {
    "alkaline_degreasing": {
        "standard": {"steel": "ASTM B183", "aluminium": "ASTM B254"},
        "parameters": {
            "steel": {
                "acid_concentration_%": {"min": 0.05, "max": 0.1},
                "water_consumption_l": {"min": 5.0, "max": 15.0},
                "temperature_C": {"min": 60, "max": 80},
                "time_min": {"min": 5, "max": 15},
            },
            "aluminium": {
                "acid_concentration_%": {"min": 0.03, "max": 0.1},
                "water_consumption_l": {"min": 3.0, "max": 10.0},
                "temperature_C": {"min": 50, "max": 70},
                "time_min": {"min": 8, "max": 20},
            },
        },
        "description": {
            "steel": "Remove oils, grease, and contaminants from steel surface",
            "aluminium": "Remove oxides and contaminants using alkaline solutions with inhibitors",
        },
    },
    "chemical_etching": {
        "standard": "ASTM B253",
        "parameters": {
            "aluminium": {
                "acid_concentration_%": {"min": 0.05, "max": 0.10},
                "water_consumption_l": {"min": 5.0, "max": 15.0},
                "time_min": {"min": 2, "max": 10},
                "temperature_C": {"min": 50, "max": 65},
            },
        },
        "description": "Alkaline (NaOH) etch of aluminium to remove the natural oxide layer and a thin surface layer",
    },
    "pickling": {
        "standard": "ASTM A380, ISO 16048",
        "parameters": {
            "steel": {
                "acid_concentration_%": {"min": .1, "max": 0.2},
                "hydrofluoric_acid_consumption_%": {"min": 0.01, "max": 0.02},  # not modelled yet
                "water_consumption_l": {"min": 10.0, "max": 25.0},
                "time_min": {"min": 10, "max": 30},
                "temperature_C": {"min": 20, "max": 50},
            },
        },
        "description": "Remove weld scale and oxides using acid solutions",
    },
    "passivation": {
        "standard": "ASTM A380, ASTM A967",
        "parameters": {
            "steel": {
                "acid_concentration_%": {"min": 0.2, "max": 0.4},
                "water_consumption_l": {"min": 5.0, "max": 10.0},
                "time_min": {"min": 20, "max": 30},
                "temperature_C": {"min": 20, "max": 50},
            },
        },
        "description": "Form protective oxide layer on stainless steel surface",
    },
}

In [ ]:
# Mechanical treatments. Each value is the total consumed over one execution.
MECH_PREPOST_PARAMS = {
    "mechanical_treatment": {
        "standard": {"steel": "ISO 8501-1 (surface preparation)", "aluminium": "ISO 8501-1"},
        "parameters": {
            "steel": {
                "abrasive_consumption_kg": {"min": 0.01, "max": 0.05},
                "electricity_kwh": {"min": 0.1, "max": 0.4},
                "time_min": {"min": 5, "max": 20},
                "surface_area_m2": {"min": 0.5, "max": 2.0},
            },
            "aluminium": {
                "abrasive_consumption_kg": {"min": 0.005, "max": 0.02},
                "coolant_consumption_l": {"min": 0.5, "max": 2.0},
                "electricity_kwh": {"min": 0.05, "max": 0.2},
                "time_min": {"min": 3, "max": 15},
                "surface_area_m2": {"min": 0.3, "max": 1.5},
            },
        },
        "condition": {"steel": "if oxides/scale are present"},
        "note": {"aluminium": "Use non-ferrous tools with coolant"},
        "description": {
            "steel": "Remove oxides and scale using abrasive methods",
            "aluminium": "Surface preparation using non-ferrous tools with mandatory coolant",
        },
    },
    "machining": {
        "standard": "ISO 3685 (tool life)",
        "parameters": {
            "steel": {
                "machining_time_min": {"min": 1, "max": 5},
                "tool_wear_pct": {"min": 5, "max": 15},
                "electricity_kwh": {"min": 0.05, "max": 0.2},
            },
        },
        "condition": "if tight tolerances are required",
        "description": "Precision machining for tight tolerance requirements",
    },
    "grinding_polishing": {
        "standard": {"steel": "ISO 16048, ASTM B912 (electropolishing)", "aluminium": "ISO 16048, ASTM B209"},
        "parameters": {
            "steel": {
                "grinding_wheel_consumption_kg": {"min": 0.005, "max": 0.02},
                "polishing_compound_kg": {"min": 0.01, "max": 0.05},
                "electricity_kwh": {"min": 0.1, "max": 0.5},
                "time_min": {"min": 5, "max": 25},
                "surface_roughness_um": {"min": 0.1, "max": 1.6},
            },
            "aluminium": {
                "grinding_wheel_consumption_kg": {"min": 0.002, "max": 0.01},
                "polishing_compound_kg": {"min": 0.005, "max": 0.02},
                "coolant_consumption_l": {"min": 0.1, "max": 0.5},
                "electricity_kwh": {"min": 0.05, "max": 0.2},
                "time_min": {"min": 3, "max": 15},
                "surface_roughness_um": {"min": 0.05, "max": 1.0},
            },
        },
        "condition": {"steel": "as needed for surface finish"},
        "note": {"aluminium": "Use non-ferrous tools with mandatory coolant"},
        "description": {
            "steel": "Achieve required surface finish and dimensional accuracy",
            "aluminium": "Final surface finishing using non-ferrous tools with coolant",
        },
    },
}

# Conversion of the mechanical parameters to ecoinvent units
COOLANT_DENSITY_KG_PER_L = 0.9
TOOL_MASS_KG = 0.01   # about 10 g carbide indexable insert (typical 5-15 g), consumed at 100 % wear

# Parameter -> (flow, factor to the ecoinvent unit). Parameters without a flow are not modelled.
MECH_PARAMETER_FLOWS = {
    "electricity_kwh": ("electricity", 1.0),
    "abrasive_consumption_kg": ("abrasive", 1.0),
    "grinding_wheel_consumption_kg": ("grinding_wheel", 1.0),
    "polishing_compound_kg": ("polishing_compound", 1.0),
    "coolant_consumption_l": ("coolant", COOLANT_DENSITY_KG_PER_L),
    "tool_wear_pct": ("tool_material", TOOL_MASS_KG / 100),   # % -> kg
}

## Project and background data

In [ ]:
ei = open_project(PROJECT_NAME, ECOINVENT_DATABASE_NAME)
flows = {name: find_activity(ECOINVENT_DATABASE_NAME, *spec) for name, spec in FLOWS.items()}
display({name: f"{a['name']} [{a['location']}] ({a['unit']})" for name, a in flows.items()})

methods = ef31_methods(INDICATORS)
units = method_units(methods)
impacts = BackgroundImpacts(methods, flows["electricity"])
prices = {flows[name].key: price for name, price in COST_FACTORS.items()}

## Chemical pre- and post-joining processes

- Active chemical (kg of active substance): concentration x bath mass x replenished fraction per hour x hours.
- Electricity (kWh): heat that replaces the tank heat loss during the process time.
- Water (kg): rinsing water.

In [ ]:
chem_data = {}
for service, material, params, _ in per_material(CHEM_PREPOST_PARAMS):
    code = process_code(service, material)
    chem_data[(CHEM_PREPOST_NAME, code)] = {
        "name": process_name(service, material), "unit": "unit", "location": "NL",
        "exchanges": chemical_exchanges(
            CHEM_PREPOST_NAME, code, params, flows[ACTIVE_CHEMICAL[service]], flows["electricity"], flows["water"],
            tank_volume_l=INDUSTRIAL_TANK_SIZES_L.get(service, DEFAULT_TANK_SIZE_L),
            replenishment_per_h=REPLENISHMENT_PER_H, bath_density_kg_per_l=BATH_DENSITY_KG_PER_L,
            room_temp_c=ROOM_TEMP_C, u_value=TANK_U_VALUE, heater_efficiency=HEATER_EFFICIENCY),
    }
chem = bd.Database(CHEM_PREPOST_NAME)
chem.write(chem_data)

## Mechanical pre- and post-joining processes

In [ ]:
parameter_flows = {p: (flows[f], factor) for p, (f, factor) in MECH_PARAMETER_FLOWS.items()}
mech_data = {}
for service, material, params, _ in per_material(MECH_PREPOST_PARAMS):
    code = process_code(service, material)
    exchanges, ignored = mechanical_exchanges(MECH_PREPOST_NAME, code, params, parameter_flows)
    print(f"{code}: not modelled (no ecoinvent flow): {ignored}")
    mech_data[(MECH_PREPOST_NAME, code)] = {
        "name": process_name(service, material), "unit": "unit", "location": "NL", "exchanges": exchanges}
mech = bd.Database(MECH_PREPOST_NAME)
mech.write(mech_data)

## LCIA results

In [ ]:
lcia_df = scenario_table([chem, mech], impacts, prices)
export(lcia_df, units, RESULTS_DIR, f"{CHEM_PREPOST_NAME}_{MECH_PREPOST_NAME}")
lcia_df

## Regional analysis: EU electricity markets

Replaces the RER electricity market with the market of each member state. The map shows the impacts and consumable costs per execution of each process.
Only electricity changes: water, chemicals and abrasives keep their ecoinvent datasets and prices.
The file `results/regional_prepost.csv` is an input of `workflow_assembly.ipynb`.

In [ ]:
markets = electricity_markets(ei, EU27)
regional_df = regional_table([chem, mech], impacts, markets, ELEC_PRICE, prices, REGIONAL_BASELINE)
regional_df.to_csv(f"{RESULTS_DIR}/regional_prepost.csv", index=False)
regional_df.head()

In [ ]:
display(plots.regional_explorer(
    regional_df,
    filters=[("Process", "process", sorted(regional_df["process"].unique()), None),
             ("Scenario", "scenario", list(SCENARIOS), "amount")],
    units={**units, COST_COLUMN: "EUR"},
    title=lambda f: f"{f['process']} ({f['scenario']}): per execution",
    baseline=REGIONAL_BASELINE,
))